# My First AI Research Assistant - ReAct Agent

In [ ]:
!pip install -q tavily-python

In [ ]:
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display
from PIL import Image
import io
import time
import os
from datetime import date
import json
from tavily import TavilyClient

---
## 🔐 SECTION 2: Load API Keys Securely

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Secure API Key Loading
#
# INDUSTRY PRACTICE: Load secrets from environment, never hardcode them.
# Colab's userdata.get() reads from the Secrets panel (key icon in sidebar).
#
# This pattern is equivalent to os.environ.get() on a production server,
# where keys are injected via CI/CD or secret managers (AWS Secrets, Vault).
# ─────────────────────────────────────────────────────────────────────────────

#Load keys from Colab Secrets into environment variables
os.environ["GEMINI_API_KEY"]  = userdata.get("Gemini_key")
os.environ["TAVILY_API_KEY"]  = userdata.get("Tavily")

# Validate that keys were loaded
gemini_key = os.environ.get("GEMINI_API_KEY", "")
tavily_key = os.environ.get("TAVILY_API_KEY", "")

---
## 🧠 SECTION 3: Initialize the LLM Brain

### Why Google Gemini 2.5 Flash Lite?
### Selected as per current Free tier models available (March 2026)

| Criterion | Gemini 2.5 Flash Lite | Alternatives |
|-----------|-----------------|-------------|
| **Cost** | Truly free (no credit card) | Groq: free but rate-limited; OpenAI: requires card |
| **Tool calling** | Native, robust | All major providers support it |
| **Context window** | 1,000,000 tokens | Groq: 128K; GPT-4o: 128K |
| **Speed** | Very fast | Groq is faster; OpenAI is slower |
| **Reliability** | Google infrastructure | Groq has occasional outages |
| **Industry use** | Growing rapidly in production | OpenAI still most common in enterprise |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — The LLM Brain (Google Gemini 2.5 Flash Lite)
#
# CONCEPT: The LLM is the reasoning engine.
#   - It reads the conversation history (memory)
#   - It reads the tool schemas (what it can do)
#   - It decides: "call a tool" OR "give the final answer"
#
# We use the google-genai SDK with OpenAI-compatible client style.
# This is the SAME interface used by Groq, Mistral, Together AI — so
# switching providers only requires changing base_url + model name.
# ─────────────────────────────────────────────────────────────────────────────


# Initialize the Gemini client
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Model selection:
#   Search up-to-date info on Free Tier models when you run this
#   gemini-2.0-flash  → best balance: speed, capability, free tier (recommended)
#   gemini-2.5-pro    → more capable but lower free tier limits (5 RPM free)
#   gemini-2.0-flash-lite → fastest and cheapest, less capable
MODEL_ID = "gemini-2.5-flash-lite"

print(f"✅ Gemini client initialized with model: {MODEL_ID}")


---
## 🔧 SECTION 4: Define the Tools

### Conceptual Model of Tools

```
  Tools are the agent's "hands".
  The LLM (brain) decides WHAT to do.
  Tools are HOW it acts in the world.

  LLM never directly runs code.
  It outputs structured JSON: { "tool": "search_web", "query": "..." }
  WE parse that and execute the real function.
  WE return the result back to the LLM.
```

### Why Tavily?

Tavily is the **default search tool in LangChain** and is referenced in **OpenAI's official agent cookbook**. It is purpose-built for LLMs — it doesn't just return URLs, it returns **clean, parsed text** ready for the model to read.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Tool Definition
#
# We define our tools in TWO places:
#   1. As Python functions  → what actually gets executed
#   2. As JSON schemas      → what the LLM reads to understand the tool
#
# The LLM reads the schema. We run the function. The loop connects them.
# ─────────────────────────────────────────────────────────────────────────────

# Initialize the Tavily search client
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# ── TOOL FUNCTION ──────────────────────────────────────────────────────────
def search_web(query: str, max_results: int = 3) -> str:
    """
    Search the web using Tavily — the industry-standard search API for AI agents.

    Unlike raw search APIs, Tavily:
      - Returns pre-extracted, clean text (not raw HTML)
      - Scores and ranks results for LLM relevance
      - Is designed to reduce hallucinations
      - Is the default tool in LangChain's TavilySearchResults

    Parameters
    ----------
    query       : The search query string
    max_results : How many results to return (1–5 recommended)

    Returns
    -------
    Formatted string with search results, ready for the LLM to consume
    """
    print(f"\n  🔍 [TOOL CALLED] search_web(query='{query}')")

    response = tavily_client.search(
        query=query,
        max_results=max_results,
        search_depth="basic"  # use "advanced" for deeper results (costs 2x credits)
    )

    # Format results into clean text for the LLM
    results_text = []
    for i, result in enumerate(response.get("results", []), 1):
        results_text.append(
            f"[Result {i}]\n"
            f"Title   : {result.get('title', 'N/A')}\n"
            f"URL     : {result.get('url', 'N/A')}\n"
            f"Content : {result.get('content', '')[:500]}..."
        )

    formatted_output = "\n\n".join(results_text) if results_text else "No results found."
    print(f"  📄 [TOOL RESULT] Returned {len(results_text)} result(s)")
    return formatted_output

# ── TOOL REGISTRY ──────────────────────────────────────────────────────────
# This dictionary maps tool name strings → actual Python functions.
# When the LLM says "call search_web", we look it up here and execute it.

TOOL_REGISTRY = {
    "search_web": search_web
}

print("✅ Tools defined:")
for name in TOOL_REGISTRY:
    print(f"   - {name}")

---
## 📐 SECTION 5: Tool Schema — Teaching the LLM What It Can Do

The LLM doesn't know Python. We must describe tools in **structured JSON format** so it understands:
- What tools are available
- What each tool does
- What parameters each tool needs

This is called **Function Calling** or **Tool Use** — a standard feature across all major LLM providers.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Tool Schema (Function Calling Definition)
#
# CONCEPT: This is the "menu" we hand to the LLM.
#          The LLM reads this and can then decide to "order" a tool call.
#
# This format follows the OpenAI function calling standard, which is now
# supported by virtually every major LLM provider. Learning this schema
# means you can use it with Gemini, OpenAI, Groq, Mistral — all the same.
# ─────────────────────────────────────────────────────────────────────────────

# Tool schemas in Gemini's types.Tool format
# (Internally identical to OpenAI's function calling JSON schema)
TOOLS = [
    types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="search_web",
                description=(
                    "Search the internet for current, real-world information about a topic. "
                    "Use this whenever you need up-to-date facts, news, data, or research "
                    "that you may not already know. Returns titles, URLs, and content snippets."
                ),
                parameters=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "query": types.Schema(
                            type=types.Type.STRING,
                            description="A specific, concise search query. Be precise — good: 'Python async await explained 2025', bad: 'Python'"
                        ),
                        "max_results": types.Schema(
                            type=types.Type.INTEGER,
                            description="Number of results to return. Use 3 for general queries, 5 for deep research."
                        )
                    },
                    required=["query"]
                )
            )
        ]
    )
]

print("✅ Tool schema defined and ready for LLM")

---
## 📝 SECTION 6: The System Prompt — Giving the Agent Its Identity

The system prompt is arguably the **most important component** of an agent. It defines:
- **Who** the agent is (persona)
- **What** it should do (task)
- **How** it should behave (constraints and style)
- **When** to use tools vs. answer directly

> **Industry Insight:** 80% of agent quality comes from prompt engineering. Senior AI engineers spend more time on system prompts than on code.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — System Prompt (Agent Identity & Behavior)
#
# DESIGN PRINCIPLES:
#   - Clear persona: what is this agent?
#   - Explicit workflow: tell the model HOW to think
#   - Output format: specify exactly what the final answer should look like
#   - Constraints: tell it what NOT to do (critical for reliability)
#
# This is the ReAct prompt pattern: Reason → Act → Observe → Repeat
# Reference: Yao et al. (2022), "ReAct: Synergizing Reasoning and Acting in Language Models"
# ─────────────────────────────────────────────────────────────────────────────

# Extract and Provide current date to agent in system prompt
# LLM model is trained up to a certain date in the past and has no understanding of what today's actual date is
today = date.today().strftime("%B %d, %Y")


SYSTEM_PROMPT = f"""
You are a Research Assistant Agent.
Today's date is {today}.
Use this to correctly interpret 'yesterday', 'recent', 'latest'.

Your job is to research topics thoroughly
and deliver clear, well-structured, factual summaries with latest information.

## Your Reasoning Process (ReAct Pattern)
You operate in a deliberate cycle — do NOT skip steps:

  STEP 1 — REASON  : What do I need to know? What gaps exist in my knowledge?
  STEP 2 — ACT     : Call search_web with a targeted query
  STEP 3 — OBSERVE : Read and interpret the search results
  STEP 4 — DECIDE  : Do I have enough info? If YES → write answer. If NO → go to Step 1.

## Search Strategy
- Always search at least 2 times with DIFFERENT angles/queries
- First search: broad overview
- Second search: specific detail, recent developments, or technical depth
- Third search (if needed): verify a specific claim or find missing context
- Make queries specific (e.g., 'MoE LLM architecture 2025' not just 'MoE')

## When You Have Enough Information
Write a structured report with EXACTLY this format:

---
📌 **Topic:** [topic name]

🔑 **Key Findings**
- [finding 1]
- [finding 2]
- [finding 3]
- [finding 4]
- [finding 5]

📖 **Detailed Summary**
[2-3 substantive paragraphs with clear explanations]

🔗 **Sources**
- [URL 1]
- [URL 2]
- [URL 3]
---

## Constraints
- NEVER fabricate facts. If unsure, say so and search again.
- NEVER stop after just one search unless the answer is trivially simple.
- ALWAYS cite your sources.
- MOST IMPORTANT constraint is to summarize your response in a meaningful way with all key details and structure in less than 1800 words.
"""

print("✅ System prompt defined")
print(f"   Length: {len(SYSTEM_PROMPT)} characters")

In [ ]:
print(SYSTEM_PROMPT)

---
## 🧩 SECTION 7: The Core Components

Two helper functions that power the agent loop:
1. `call_llm()` — sends the conversation to Gemini, gets back a decision
2. `execute_tool_call()` — runs the tool the LLM requested

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7A — The LLM Caller (The "Think" Step)
#
# WHAT HAPPENS HERE:
#   1. We send the full conversation history to the LLM
#   2. The LLM reads the system prompt + all past messages + tool schemas
#   3. It responds with EITHER:
#      a) A tool_call  →  "I want to call search_web({query: '...'})"
#      b) A text reply →  "Here is the final answer: ..."
# ─────────────────────────────────────────────────────────────────────────────

def call_llm(conversation_history: list) -> object:
    """
    Send the current conversation history to the LLM and get its next response.

    Parameters
    ----------
    conversation_history : list of dicts
        Full conversation so far, in Gemini's Content format.
        Each item has 'role' (user/model) and 'parts'.

    Returns
    -------
    The LLM response object — either contains tool_calls or text.
    """
    response = gemini_client.models.generate_content(
        model=MODEL_ID,
        contents=conversation_history,   # ← full history = short-term memory
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,   # ← agent identity
            tools=TOOLS,                         # ← tool schemas available
            temperature=0.2,                     # ← low = factual, focused
            max_output_tokens=2048
        )
    )
    return response


print("✅ call_llm() defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7B — The Tool Executor (The "Act" Step)
#
# WHAT HAPPENS HERE:
#   1. The LLM's response contains a function_call with a tool name + arguments
#   2. We look up the tool name in our TOOL_REGISTRY
#   3. We execute the actual Python function with the given arguments
#   4. We return the result in the format Gemini expects back
#
# IMPORTANT CONCEPT: The LLM is the DECIDER. We are the EXECUTOR.
# The LLM cannot run Python. It can only request that we run it.
# This separation is what makes agents safe and auditable.
# ─────────────────────────────────────────────────────────────────────────────

def execute_tool_call(function_call) -> types.Part:
    """
    Execute the tool that the LLM requested and return the result.

    Parameters
    ----------
    function_call : The function_call part from the LLM's response
        Contains .name (tool name) and .args (dict of arguments)

    Returns
    -------
    types.Part : A FunctionResponse part to add back into the conversation.
                 The LLM will read this as the "Observe" step in ReAct.
    """
    tool_name = function_call.name
    tool_args = dict(function_call.args)  # convert MapComposite → regular dict

    print(f"\n  ⚙️  [EXECUTE] {tool_name}({tool_args})")

    try:
        if tool_name in TOOL_REGISTRY:
            result = TOOL_REGISTRY[tool_name](**tool_args)
        else:
            result = f"Error: Tool '{tool_name}' not found in registry."
            print (result)
    except Exception as e:
        result = f"Tool execution failed: {type(e).__name__}: {str(e)}"
        print (result)


    # Return result in Gemini's FunctionResponse format
    # This gets added to the conversation so the LLM can "observe" it
    return types.Part.from_function_response(
        name=tool_name,
        response={"result": result}
    )


print("✅ execute_tool_call() defined")

---
## 🔄 SECTION 8: The Agent Loop — The Heart of Every Agent

This is what transforms an LLM into an **agent**. The key insight:

> A single LLM call is a **chatbot**.  
> An LLM in a loop with tools and memory is an **agent**.

```
START
  │
  ▼
call_llm(history)
  │
  ├─ LLM wants a tool ──► execute_tool ──► append result to history ──► loop back
  │
  └─ LLM gives answer ──► DONE
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — The Agent Loop (ReAct Implementation)
#
# This is the most important function in the notebook.
# Everything else is setup. This is where the agent LIVES.
#
# THE LOOP:
#   while True:
#     response = call_llm(history)          ← REASON
#     if response has tool_calls:
#       result = execute_tool(tool_call)    ← ACT
#       history.append(result)             ← OBSERVE (update memory)
#     else:
#       return response.text               ← FINAL ANSWER
#
# The agent DECIDES when to stop. We only set a max_iterations safety limit.
# This is what makes it autonomous — not a scripted pipeline.
# ─────────────────────────────────────────────────────────────────────────────

def run_agent(user_query: str, max_iterations: int = 2) -> str:
    """
    Run the full ReAct agent loop for a given research query.

    The agent autonomously decides:
      - How many times to search
      - What queries to use
      - When it has enough information to answer

    Parameters
    ----------
    user_query     : The research question from the user
    max_iterations : Safety cap on loop iterations (prevents runaway agents)

    Returns
    -------
    str : The agent's final structured report
    """

    print("\n" + "═" * 65)
    print(f"  🚀 AGENT STARTED")
    print(f"  📋 Query: {user_query}")
    print("═" * 65)

    # ── MEMORY INITIALIZATION ──────────────────────────────────────────────
    # Short-term memory = the conversation history list.
    # On every LLM call, we pass the FULL history.
    # This is how the agent "remembers" searches it already did,
    # results it already saw, and reasoning it already did.
    #
    # This is the simplest form of agent memory — "in-context memory".
    # Production agents add long-term memory via vector databases (ChromaDB,
    # Pinecone, etc.), but in-context memory is the foundation.
    # ──────────────────────────────────────────────────────────────────────
    conversation_history = [
        types.Content(
            role="user",
            parts=[types.Part(text=user_query)]
        )
    ]

    # ── THE AGENT LOOP ─────────────────────────────────────────────────────
    for iteration in range(1, max_iterations + 1):
        print(f"\n{'─' * 65}")
        print(f"  🔄 ITERATION {iteration} — Calling LLM...")
        print(f"{'─' * 65}")

        # ── REASON: Ask the LLM what to do next ────────────────────────────
        response = call_llm(conversation_history)
        candidate = response.candidates[0]
        content   = candidate.content

        # Check each part of the response — could be text or tool calls
        has_tool_call = False
        tool_response_parts = []

        for part in content.parts:

            # ── ACT: LLM wants to call a tool ──────────────────────────────
            if part.function_call:
                has_tool_call = True
                print(f"\n  🤔 [LLM DECISION] Call tool: '{part.function_call.name}'")

                # Execute the tool and collect the result
                tool_response = execute_tool_call(part.function_call)
                tool_response_parts.append(tool_response)

        if has_tool_call:
            # ── OBSERVE: Append the assistant's decision + tool results ────
            # Add the LLM's message (which contained the tool_call request)
            conversation_history.append(content)

            # Add the tool execution results so LLM can read them next round
            conversation_history.append(
                types.Content(
                    role="user",
                    parts=tool_response_parts
                )
            )
            # Loop back — LLM will now read the results and decide next step
            continue

        else:
            # ── FINAL ANSWER: LLM returned text without calling a tool ─────
            final_text = "".join(
                part.text for part in content.parts if hasattr(part, "text")
            )

            print(f"\n  ✅ [LLM DECISION] Provide final answer")
            print("\n" + "═" * 65)
            print("  📊 FINAL REPORT")
            print("═" * 65)
            print(final_text)
            print("═" * 65)
            print(f"\n  ✔  Completed in {iteration} iteration(s)")
            return final_text

    # Safety: max iterations reached
    return f"⚠️ Agent reached max iterations ({max_iterations}) without a final answer."


print("✅ run_agent() defined — Agent is ready!")

---
## 🚀 SECTION 9: Run the Agent!

Now we put it all together. The agent will:
1. Receive a research question
2. Reason about what it needs to know
3. Search the web 2–3 times with different queries
4. Synthesize the results
5. Return a structured report

All autonomously. No scripted steps.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — Run the Agent
#
# Try changing the topic below to research anything you're curious about.
# The agent will autonomously decide how many searches to perform.
# ─────────────────────────────────────────────────────────────────────────────


# Short simple question for TESTING purpose to preserve computing cost and api calls
# It also helps check if the AI agent is able to get real time and latest information
research_topic = """
What was the India-England match score from yesterday's match?
"""

# ── Run the Agent ──────────────────────────────────────────────────────────
result = run_agent(research_topic)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — Run the Agent
#
# Try changing the topic below to research anything you're curious about.
# The agent will autonomously decide how many searches to perform.
# ─────────────────────────────────────────────────────────────────────────────

# ── Research Topic ─────────────────────────────────────────────────────────
# This is a good test topic because it requires:
#   - Multiple searches (broad + deep)
#   - Recent information (post-training-cutoff)
#   - Technical synthesis

research_topic = """
What is Mixture of Experts (MoE) architecture in LLMs?
Why is it important, and which major models use it as of 2025?
"""


# ── Run the Agent ──────────────────────────────────────────────────────────
result = run_agent(research_topic)

---
## 🧪 SECTION 10: Experiment — Try Different Topics

Run the agent on different topics to observe how it:
- Adjusts search strategy per topic
- Decides how many iterations to run
- Structures its output differently

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 — Experiments
#
# Try each of these topics and observe the agent's behavior.
# Compare: How many iterations? How many searches? How different is the output?
# Try changing System Prompt and see how LLM behaves differently
# ─────────────────────────────────────────────────────────────────────────────

# --- EXPERIMENT 1: Simple factual query (agent may answer in 1-2 iterations) ---
simple_query = "What is Retrieval Augmented Generation (RAG) and how does it work?"

# --- EXPERIMENT 2: Complex current events (agent will need 3+ searches) ---
complex_query = "What are the key differences between GPT-4o, Gemini 2.5 Pro, and Claude 3.5 Sonnet?"

# --- EXPERIMENT 3: Technical + recent (tests both depth and recency) ---
technical_query = "What is the Model Context Protocol (MCP) and why is the AI industry adopting it rapidly in 2025?"

# ── Uncomment ONE to run ────────────────────────────────────────────────────
# result2 = run_agent(simple_query)
# result3 = run_agent(complex_query)
# result4 = run_agent(technical_query)

print("💡 Uncomment one of the lines above and run this cell to experiment.")

---
## 💡 SECTION 11: Key Takeaways

### What You Just Built
A complete ReAct agent from scratch using 100% free tools — no frameworks, no Langchain, no MCP, no black boxes.

### The 5 Building Blocks Recap

| Component | Key Insight |
|-----------|------------|
| **LLM Brain** | The brain doesn't run code. It only reasons and requests. |
| **Tools** | Tools are just Python functions. The schema teaches the LLM how to request them. |
| **Agent Loop** | Loop = autonomy. A single call = chatbot. A loop with tools = agent. |
| **Memory** | Short-term memory = the conversation history list passed every iteration. |
| **System Prompt** | The most powerful lever. Defines WHO the agent is and HOW it thinks. |


